In [1]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet
Shape: (12032, 15)


,src,answer,options,category,question,cot_content,question_id,answer_index,total_tokens,meta_cluster,base_cluster,distill_ans_correct,distill_reasoning,distill_answer,corrected_reasoning
0,ori_mmlu-jurisprudence,C,['There is no distinction between the two form...,law,Which of the following criticisms of Llewellyn...,NaN,1286,2,81,Legal Interpretation,Legal Theory Interpretations,True,"We are asked: ""Which of the following criticis...",c,
1,ori_mmlu-international_law,E,"['Article 19', 'Article 11', 'Article 12', 'Ar...",law,Which of the following articles are not qualif...,NaN,1293,4,38,Legal Interpretation,Constitutional Law,True,"We are asked: ""Which of the following articles...",e,


## Accuracy


In [2]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 1.0000 (12032/12032)


## Missing distilled reasoning traces


In [3]:
missing_nan = int(df["distill_reasoning"].isna().sum())
empty_mask = df["distill_reasoning"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 0


## Reasoning trace length distribution (chars)


In [4]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
existing_reasoning.str.len().describe()

count     12032.000000
mean       7926.306433
std       17770.740458
min          76.000000
25%         934.000000
50%        2055.000000
75%        5984.250000
max      214703.000000
Name: distill_reasoning, dtype: float64

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [5]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


n_opts_series = df["options"].apply(n_options)
valid_mask = pd.Series(
    [is_valid_answer(a, k) for a, k in zip(df["distill_answer"], n_opts_series)],
    index=df.index,
)
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 0 (0.00%)


### All invalid answers


In [6]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

### 10 shortest reasoning traces


In [7]:
shortest = existing_reasoning.str.len().nsmallest(10)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

--- row 4440 (len=76) ---
We need to find three fifths of 100. That is (3/5)*100 = 60. So answer is f.

--- row 5089 (len=89) ---
We need to find the product of 5 and -9. 5 * (-9) = -45. So the correct option is b. −45.

--- row 6308 (len=93) ---
We need to solve 5x - 5 = -10. Add 5 to both sides: 5x = -5, then x = -1. So answer is h. -1.

--- row 6099 (len=94) ---
We need to evaluate log base 3 of 81. 81 is 3^4, so log_3(81) = 4. The correct option is e. 4.

--- row 3803 (len=99) ---
We need to find which exponential notation equals 343. Compute 7^3 = 7*7*7 = 343. So answer b. 7^3.

--- row 2107 (len=102) ---
We are asked: "What is the value of y in the equation y/4 = 8?" So y = 8 * 4 = 32. So answer is g. 32.

--- row 8493 (len=106) ---
We need to select the correct option. Lewis and Clark: Meriwether Lewis and William Clark. So answer is h.

--- row 1080 (len=109) ---
We need to compute 32 * 67. Let's calculate: 30 * 67 = 2010, 2 * 67 = 134, sum = 2144. So answer is g. 2,144.

--- r

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [8]:
# invalid_idx = df.index[~valid_mask]
# df.loc[invalid_idx, "distill_reasoning"] = ""
# df.loc[invalid_idx, "distill_answer"] = ""
# df.loc[invalid_idx, "distill_ans_correct"] = False

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [9]:
# short_mask = df["distill_reasoning"].fillna("").str.len() < 200
# short_idx = df.index[short_mask]
# df.loc[short_idx, "distill_reasoning"] = ""

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")